In [3]:
import numpy as np

In [4]:
import pandas as pd

df = pd.read_csv('../data/raw/results.csv')
print(df.shape)
df.head()

(49477, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [5]:
df.dtypes

date              str
home_team         str
away_team         str
home_score    float64
away_score    float64
tournament        str
city              str
country           str
neutral          bool
dtype: object

In [6]:
df['home_score'].isna().sum()

np.int64(52)

In [7]:
df['away_score'].isna().sum()

np.int64(52)

In [8]:
df[df['home_score'].isna()]

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49425,2026-06-17,Portugal,DR Congo,NaN,NaN,FIFA World Cup,Houston,United States,True
49426,2026-06-17,Uzbekistan,Colombia,NaN,NaN,FIFA World Cup,Mexico City,Mexico,True
49427,2026-06-17,England,Croatia,NaN,NaN,FIFA World Cup,Arlington,United States,True
49428,2026-06-17,Ghana,Panama,NaN,NaN,FIFA World Cup,Toronto,Canada,True
49429,2026-06-18,Czech Republic,South Africa,NaN,NaN,FIFA World Cup,Atlanta,United States,True
49430,2026-06-18,Mexico,South Korea,NaN,NaN,FIFA World Cup,Zapopan,Mexico,False
49431,2026-06-18,Switzerland,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,Inglewood,United States,True
49432,2026-06-18,Canada,Qatar,NaN,NaN,FIFA World Cup,Vancouver,Canada,False
49433,2026-06-19,Scotland,Morocco,NaN,NaN,FIFA World Cup,Foxborough,United States,True
49434,2026-06-19,Brazil,Haiti,NaN,NaN,FIFA World Cup,Philadelphia,United States,True


In [9]:
df['date'] = pd.to_datetime(df['date'])

In [10]:
df.dtypes

date          datetime64[us]
home_team                str
away_team                str
home_score           float64
away_score           float64
tournament               str
city                     str
country                  str
neutral                 bool
dtype: object

In [11]:
partidos_a_predecir = df[df['home_score'].isna()].copy()
df_historico = df[df['home_score'].notna()].copy()

print(partidos_a_predecir.shape)
print(df_historico.shape)

(52, 9)
(49425, 9)


In [12]:
df_historico.dtypes

date          datetime64[us]
home_team                str
away_team                str
home_score           float64
away_score           float64
tournament               str
city                     str
country                  str
neutral                 bool
dtype: object

In [13]:
partidos_argentina = df_historico[(df_historico['home_team'] == 'Argentina') | (df_historico['away_team'] == 'Argentina')]

In [14]:
print(partidos_argentina.shape)

(1070, 9)


In [15]:
partidos_argentina.tail()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49147,2026-03-27,Argentina,Mauritania,2.0,1.0,Friendly,Buenos Aires,Argentina,False
49217,2026-03-31,Argentina,Zambia,5.0,0.0,Friendly,Buenos Aires,Argentina,False
49341,2026-06-06,Argentina,Honduras,2.0,0.0,Friendly,College Station,United States,True
49380,2026-06-09,Argentina,Iceland,3.0,0.0,Friendly,Auburn,United States,True
49423,2026-06-16,Argentina,Algeria,3.0,0.0,FIFA World Cup,Kansas City,United States,True


In [16]:
partidos_argentina['goles_argentina'] = np.where(
    partidos_argentina['home_team'] == 'Argentina',  # condición
    partidos_argentina['home_score'],                 # si es local → tomá home_score
    partidos_argentina['away_score']                  # si es visitante → tomá away_score
)

In [17]:
partidos_argentina['goles_argentina'].mean()

np.float64(1.8981308411214954)

In [18]:
goles_local = df_historico.groupby('home_team')['home_score'].sum()
goles_visitante = df_historico.groupby('away_team')['away_score'].sum()

goles_totales = goles_local.add(goles_visitante, fill_value=0)

In [19]:
print(goles_totales.sort_values(ascending=False).head(10))

home_team
England        2381.0
Germany        2327.0
Brazil         2306.0
Sweden         2176.0
Argentina      2031.0
Hungary        2011.0
Netherlands    1843.0
South Korea    1794.0
Mexico         1769.0
France         1718.0
dtype: float64


In [20]:
partidos_local = df_historico.groupby('home_team').size()
partidos_visitante = df_historico.groupby('away_team').size()
partidos_totales = partidos_local.add(partidos_visitante, fill_value=0)

print(partidos_totales.sort_values(ascending=False).head(10))

home_team
Sweden         1102.0
England        1090.0
Argentina      1070.0
Brazil         1060.0
Germany        1032.0
South Korea    1008.0
Hungary        1006.0
Mexico         1004.0
Uruguay         971.0
France          936.0
dtype: float64


In [21]:
promedio_goles_totales = goles_totales / partidos_totales

print(promedio_goles_totales.sort_values(ascending=False).head(10))

home_team
Quebec                8.000000
Elba Island           4.500000
Yorkshire             3.857143
Parishes of Jersey    3.666667
Cascadia              3.285714
Isle of Man           3.206897
Provence              3.173913
Occitania             3.121212
Sápmi                 3.103448
East Turkestan        3.000000
dtype: float64


In [22]:
promedio_goles_totales[partidos_totales >= 50].sort_values(ascending=False).head(10)

home_team
Isle of Man       3.206897
Jersey            2.744681
Tahiti            2.714876
New Caledonia     2.633962
Guernsey          2.566667
Basque Country    2.562500
Fiji              2.268657
Germany           2.254845
England           2.184404
Brazil            2.175472
dtype: float64

In [23]:
condiciones = [
    df_historico['home_score'] > df_historico['away_score'],
    df_historico['home_score'] == df_historico['away_score'],
    df_historico['home_score'] < df_historico['away_score']
]

valores = ['home_win', 'draw', 'away_win']

df_historico['resultado'] = np.select(condiciones, valores, default='draw')

In [24]:
df_historico['resultado'].value_counts()

resultado
home_win    24222
away_win    13962
draw        11241
Name: count, dtype: int64

In [25]:
df_elo = pd.read_csv('../data/raw/eloratings.csv')

In [26]:
df_elo.head()



,date,team,rating,change
0,1872-11-30,England,2003.0,3
1,1872-11-30,Scotland,1997.0,-3
2,1873-03-08,England,2014.0,11
3,1873-03-08,Scotland,1986.0,-11
4,1874-03-07,England,2006.0,-8


In [27]:
df_elo.dtypes

date          str
team          str
rating    float64
change      int64
dtype: object

In [28]:
df_elo.shape

(6678, 4)

In [29]:
df_elo.isna().sum()

date       0
team       0
rating    31
change     0
dtype: int64

In [30]:
df_elo.loc[df_elo['rating'].isna()]

,date,team,rating,change
1219,10/14/1992,Moldova,NaN,-4
1346,9/6/1995,Moldova,NaN,-7
1536,10/7/1997,Moldova,NaN,-19
1775,9/1/2001,Moldova,NaN,15
1824,2/13/2002,Moldova,NaN,-32
1891,8/21/2002,Moldova,NaN,-6
2137,9/4/2004,Moldova,NaN,-10
2234,6/4/2005,Moldova,NaN,-10
2380,9/7/2005,Moldova,NaN,-15
2706,10/17/2007,Moldova,NaN,15


In [31]:
df_elo.loc[df_elo['team'] == 'Moldova']

,date,team,rating,change
1219,10/14/1992,Moldova,NaN,-4
1346,9/6/1995,Moldova,NaN,-7
1536,10/7/1997,Moldova,NaN,-19
1775,9/1/2001,Moldova,NaN,15
1824,2/13/2002,Moldova,NaN,-32
1891,8/21/2002,Moldova,NaN,-6
2137,9/4/2004,Moldova,NaN,-10
2234,6/4/2005,Moldova,NaN,-10
2380,9/7/2005,Moldova,NaN,-15
2706,10/17/2007,Moldova,NaN,15


In [32]:
df_elo['date'] = pd.to_datetime(df_elo['date'], format='mixed')

In [33]:
df_elo.dtypes

date      datetime64[us]
team                 str
rating           float64
change             int64
dtype: object

In [34]:
df_elo['date'].is_monotonic_increasing

True

In [35]:
df_historico['date'].is_monotonic_increasing

False

In [36]:
df_historico = df_historico.sort_values('date').reset_index(drop=True)

In [37]:
df_historico['date'].is_monotonic_increasing

True

In [38]:
df_elo_home = df_elo.rename(columns={'team': 'home_team', 'rating': 'home_elo', 'change': 'home_elo_change'})

df_merged = pd.merge_asof(
    df_historico,
    df_elo_home[['date', 'home_team', 'home_elo']],
    on='date',
    by='home_team'
)

In [39]:
df_merged.head()   

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,resultado,home_elo
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,draw,1997.0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,home_win,2014.0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,home_win,1994.0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,draw,2003.0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,home_win,2010.0


In [40]:
df_elo_away = df_elo.rename(columns={'team': 'away_team', 'rating': 'away_elo', 'change': 'away_elo_change'})

df_merged = pd.merge_asof(
    df_merged,
    df_elo_away[['date', 'away_team', 'away_elo']],
    on='date',
    by='away_team'
)

In [41]:
df_merged.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,resultado,home_elo,away_elo
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,draw,1997.0,2003.0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,home_win,2014.0,1986.0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,home_win,1994.0,2006.0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,draw,2003.0,1997.0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,home_win,2010.0,1990.0


In [42]:
df_merged['elo_diff'] = df_merged['home_elo'] - df_merged['away_elo']

In [43]:
df_merged.groupby('resultado')['elo_diff'].mean()

resultado
away_win   -166.014775
draw        -20.343275
home_win    134.263109
Name: elo_diff, dtype: float64

In [44]:
mini = pd.DataFrame({
    'date': ['2024-01-01', '2024-01-02'],
    'home_team': ['Argentina', 'Brasil'],
    'away_team': ['Brasil', 'Argentina'],
    'home_points': [3, 0],
    'away_points': [0, 3]
})

In [45]:
local = mini[['date', 'home_team', 'home_points']].rename(
    columns={'home_team': 'equipo', 'home_points': 'puntos'}
)
visitante = mini[['date', 'away_team', 'away_points']].rename(
    columns={'away_team': 'equipo', 'away_points': 'puntos'}
)

partido_por_equipo = pd.concat([local, visitante]).sort_values('date').reset_index(drop=True)

In [46]:

partido_por_equipo.head()

,date,equipo,puntos
0,2024-01-01,Argentina,3
1,2024-01-01,Brasil,0
2,2024-01-02,Brasil,0
3,2024-01-02,Argentina,3


In [47]:
df_merged['home_points'] = np.select(
    [df_merged['resultado'] == 'home_win',
     df_merged['resultado'] == 'draw', 
        df_merged['resultado'] == 'away_win'],
    [3, 1, 0]
)

In [48]:
df_merged['away_points'] = np.select(
    [df_merged['resultado'] == 'away_win',
     df_merged['resultado'] == 'draw', 
        df_merged['resultado'] == 'home_win'],
    [3, 1, 0]
)

In [49]:
local = df_merged[['date', 'home_team', 'home_points', 'elo_diff']].rename(
    columns={'home_team': 'equipo', 'home_points': 'puntos'}
)

In [50]:
visitante = df_merged[['date', 'away_team', 'away_points', 'elo_diff']].rename(
    columns={'away_team': 'equipo', 'away_points': 'puntos'}
)

In [51]:
df_forma = pd.concat([local, visitante]).sort_values('date').reset_index(drop=True)
df_forma.head()

,date,equipo,puntos,elo_diff
0,1872-11-30,Scotland,1,-6.0
1,1872-11-30,England,1,-6.0
2,1873-03-08,Scotland,0,28.0
3,1873-03-08,England,3,28.0
4,1874-03-07,Scotland,3,-12.0


In [52]:
df_forma['forma_reciente'] = (
    df_forma.groupby('equipo')['puntos']
    .rolling(5, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

In [53]:
df_forma.head()

,date,equipo,puntos,elo_diff,forma_reciente
0,1872-11-30,Scotland,1,-6.0,1.0
1,1872-11-30,England,1,-6.0,1.0
2,1873-03-08,Scotland,0,28.0,1.0
3,1873-03-08,England,3,28.0,4.0
4,1874-03-07,Scotland,3,-12.0,4.0


In [57]:
forma_home = df_forma[['date', 'equipo', 'forma_reciente']].rename(
    columns={'equipo': 'home_team', 'forma_reciente': 'home_forma'}
)

In [58]:
df_merged = df_merged.merge(forma_home, on=['date', 'home_team'], how='left')

In [59]:
forma_away = df_forma[['date', 'equipo', 'forma_reciente']].rename(
    columns={'equipo': 'away_team', 'forma_reciente': 'forma_away'}
)

In [60]:
df_merged = df_merged.merge(forma_away, on=['date', 'away_team'], how='left')

In [61]:
df_merged = df_merged.rename(columns={'forma_away': 'away_forma'})

In [62]:
df_merged = df_merged.rename(columns={'forma_away': 'away_forma'})
df_merged.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,resultado,home_elo,away_elo,elo_diff,home_points,away_points,home_forma,away_forma
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,draw,1997.0,2003.0,-6.0,1,1,1.0,1.0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,home_win,2014.0,1986.0,28.0,3,0,4.0,1.0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,home_win,1994.0,2006.0,-12.0,3,0,4.0,4.0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,draw,2003.0,1997.0,6.0,1,1,5.0,5.0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,home_win,2010.0,1990.0,20.0,3,0,8.0,5.0


In [63]:
mini = pd.DataFrame({
    'home_team': ['Argentina', 'Brasil', 'Chile'],
    'away_team': ['Brasil', 'Argentina', 'Peru']
})

mini['equipo_a'] = mini[['home_team', 'away_team']].min(axis=1)
mini['equipo_b'] = mini[['home_team', 'away_team']].max(axis=1)

mini

,home_team,away_team,equipo_a,equipo_b
0,Argentina,Brasil,Argentina,Brasil
1,Brasil,Argentina,Argentina,Brasil
2,Chile,Peru,Chile,Peru


In [64]:
df_merged['equipo_a'] = df_merged[['home_team', 'away_team']].min(axis=1)
df_merged['equipo_b'] = df_merged[['home_team', 'away_team']].max(axis=1)

In [65]:
df_merged.groupby(['equipo_a', 'equipo_b']).size().sort_values(ascending=False).head(10)

equipo_a   equipo_b   
Argentina  Uruguay        208
Austria    Hungary        141
Belgium    Netherlands    129
Guernsey   Jersey         119
England    Scotland       118
Norway     Sweden         115
Argentina  Brazil         114
Denmark    Sweden         110
Kenya      Uganda         110
Argentina  Paraguay       109
dtype: int64

In [66]:
gano_local = df_merged['resultado'] == 'home_win'
local_es_a = df_merged['home_team'] == df_merged['equipo_a']
gano_visitante = df_merged['resultado'] == 'away_win'
visitante_es_a = df_merged['away_team'] == df_merged['equipo_a']

df_merged['gano_equipo_a'] = np.where(
    (gano_local & local_es_a) | (gano_visitante & visitante_es_a),
    1, 0
)

df_merged['h2h_victorias_a'] = (
    df_merged.groupby(['equipo_a', 'equipo_b'])['gano_equipo_a']
    .transform(lambda x: x.shift(1).cumsum())
    .fillna(0)
)

In [67]:
df_merged[(df_merged['equipo_a'] == 'Argentina') & (df_merged['equipo_b'] == 'Brasil')][['date', 'home_team', 'away_team', 'resultado', 'h2h_victorias_a']].tail(10)

,date,home_team,away_team,resultado,h2h_victorias_a


In [68]:
df_merged['equipo_a'].unique()[:20]

<StringArray>
[         'England',         'Scotland', 'Northern Ireland',
           'Canada',        'Argentina',          'Austria',
   'Czechoslovakia',          'Belgium',           'France',
         'Alderney',         'Guernsey',           'Guyana',
          'Germany',           'Norway',          'Denmark',
      'Netherlands',          'Hungary',            'Chile',
            'Italy',          'Finland']
Length: 20, dtype: str

In [69]:
df_merged[df_merged['equipo_a'] == 'Argentina'].shape

(1097, 21)

In [70]:
df_merged[df_merged['equipo_b'] == 'Argentina'].shape

(5, 21)

In [71]:
df_merged[(df_merged['equipo_a'] == 'Argentina') & (df_merged['equipo_b'] == 'Brazil')][['date', 'home_team', 'away_team', 'resultado', 'h2h_victorias_a']].tail(10)

,date,home_team,away_team,resultado,h2h_victorias_a
39659,2015-11-13,Argentina,Brazil,draw,37.0
40586,2016-11-10,Brazil,Argentina,home_win,37.0
41004,2017-06-09,Argentina,Brazil,home_win,37.0
42320,2018-10-16,Argentina,Brazil,away_win,38.0
43048,2019-07-02,Brazil,Argentina,home_win,38.0
43564,2019-11-15,Brazil,Argentina,away_win,38.0
44567,2021-07-10,Brazil,Argentina,away_win,39.0
45076,2021-11-16,Argentina,Brazil,draw,40.0
47159,2023-11-21,Brazil,Argentina,away_win,40.0
48619,2025-03-25,Argentina,Brazil,home_win,41.0
